In [ ]:
# 1. Configuration: edit this cell only before the first run.
from pathlib import Path
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys

RUN_ALL = True
RESUME = True
RUN_PROFILE = "OFFICIAL_DEMO_QUICK"
SUPPORTED_PROFILES = {"OFFICIAL_DEMO_QUICK", "FORMAL_FULL"}
PREFETCH_MODEL_SNAPSHOTS = True

if RUN_PROFILE not in SUPPORTED_PROFILES:
    raise ValueError("Unsupported RUN_PROFILE: " + RUN_PROFILE)
OFFICIAL_DEMO_QUICK = RUN_PROFILE == "OFFICIAL_DEMO_QUICK"
FULL_MODE = RUN_PROFILE == "FORMAL_FULL"
RUN_MET3R = FULL_MODE

GITHUB_REPO = "https://github.com/WANG-Ruipeng/MV-Adapter.git"
GIT_BRANCH = "main"
MET3R_REVISION = "ee0e1752898559e1a3e85e2e151d3edeb9b55f73"
COLAB_REPO = Path("/content/MV-Adapter")
DRIVE_BASE = Path("/content/drive/MyDrive/Colab_Projects/MV-Adapter")
if OFFICIAL_DEMO_QUICK:
    DRIVE_PROJECT_ROOT = DRIVE_BASE / "nile_lowrank_official_demo_quick"
    INPUT_DIR = DRIVE_PROJECT_ROOT / "inputs"
else:
    DRIVE_PROJECT_ROOT = DRIVE_BASE / "nile_lowrank_full"
    INPUT_DIR = DRIVE_BASE / "nile_lowrank_full_inputs"
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "artifacts"
HF_CACHE_DIR = DRIVE_BASE / "hf_cache"
FROZEN_CONFIG = (
    DRIVE_PROJECT_ROOT / "configs" / (RUN_PROFILE.lower() + "_resolved.yaml")
)
CHECKPOINT_MANIFEST = DRIVE_PROJECT_ROOT / "configs" / "checkpoint_manifest.json"

if FULL_MODE and not RUN_MET3R:
    raise ValueError("FULL_MODE requires RUN_MET3R=True for a formal result.")

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def run_command(command, *, cwd=None, check=True, capture_output=False):
    command = [str(item) for item in command]
    print("\n$ " + shlex.join(command), flush=True)
    result = subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        check=False,
        text=True,
        capture_output=capture_output,
        env=os.environ.copy(),
    )
    if capture_output:
        if result.stdout:
            print(result.stdout.rstrip())
        if result.stderr:
            print(result.stderr.rstrip(), file=sys.stderr)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(
            result.returncode, command, result.stdout, result.stderr
        )
    return result

def runner_command(stage, *, dry_run=False):
    command = [
        sys.executable,
        "-m",
        "scripts.run_nile_lowrank_study",
        "--config",
        str(FROZEN_CONFIG),
        "--stage",
        str(stage),
        "--input-dir",
        str(INPUT_DIR),
        "--output-root",
        str(OUTPUT_ROOT),
    ]
    if RESUME:
        command.append("--resume")
    if dry_run:
        command.append("--dry-run")
    return command

def run_stage(stage, *, enabled=True, dry_run=False, capture_output=False):
    if not RUN_ALL or not enabled:
        print("Skipped stage:", stage)
        return None
    return run_command(
        runner_command(stage, dry_run=dry_run),
        cwd=COLAB_REPO,
        capture_output=capture_output,
    )

print("Run profile:", RUN_PROFILE)
print("Formal run:", FULL_MODE)
print("Artifact parent passed to runner:", OUTPUT_ROOT)
print("Input directory:", INPUT_DIR)
if OFFICIAL_DEMO_QUICK:
    print("Quick matrix: 3 official images x 1 seed x 5 methods = 15 runs.")
    print("This is a visual reproduction check, not a formal FULL result.")
print("Recommended formal GPU: A100 40GB. L4 is supported but slower.")


In [ ]:
# 2. Mount Google Drive.
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook must use a Google Colab kernel (including VS Code Colab)."
    ) from exc

drive.mount("/content/drive", force_remount=False)
for directory in (
    DRIVE_PROJECT_ROOT,
    INPUT_DIR,
    OUTPUT_ROOT,
    HF_CACHE_DIR,
    FROZEN_CONFIG.parent,
):
    directory.mkdir(parents=True, exist_ok=True)
print("Drive mounted; persistent directories are ready.")


In [ ]:
# 3. Clone or safely update the repository without overwriting dirty work.
if not (COLAB_REPO / ".git").is_dir():
    run_command(
        ["git", "clone", "--branch", GIT_BRANCH, "--single-branch",
         GITHUB_REPO, str(COLAB_REPO)]
    )
else:
    origin = run_command(
        ["git", "remote", "get-url", "origin"], cwd=COLAB_REPO,
        capture_output=True
    ).stdout.strip()
    dirty = run_command(
        ["git", "status", "--porcelain"], cwd=COLAB_REPO,
        capture_output=True
    ).stdout.strip()
    print("Existing origin:", origin)
    if dirty:
        print("Repository is dirty; preserving it exactly and skipping update.")
        print(dirty)
    else:
        run_command(["git", "fetch", "origin", GIT_BRANCH], cwd=COLAB_REPO)
        run_command(["git", "checkout", GIT_BRANCH], cwd=COLAB_REPO)
        run_command(
            ["git", "pull", "--ff-only", "origin", GIT_BRANCH],
            cwd=COLAB_REPO
        )

os.chdir(COLAB_REPO)
if str(COLAB_REPO) not in sys.path:
    sys.path.insert(0, str(COLAB_REPO))

required_experiment_files = [
    Path("configs/nile_lowrank_full.yaml"),
    Path("scripts/run_nile_lowrank_study.py"),
    Path("scripts/nile_lowrank_inference_worker.py"),
    Path("scripts/eval_nile_lowrank_study.py"),
]
missing_experiment_files = [
    str(path) for path in required_experiment_files
    if not (COLAB_REPO / path).is_file()
]
if missing_experiment_files:
    raise RuntimeError(
        "The selected remote branch does not contain this formal low-rank "
        "experiment implementation. Commit and push the local changes to "
        + GITHUB_REPO + " on branch " + GIT_BRANCH
        + ", then restart Run All. Missing files: "
        + ", ".join(missing_experiment_files)
    )
OFFICIAL_DEMO_RELATIVE_PATHS = (
    Path("assets/demo/i2mv/A_decorative_figurine_of_a_young_anime-style_girl.png"),
    Path("assets/demo/i2mv/A_juvenile_emperor_penguin_chick.png"),
    Path("assets/demo/i2mv/A_striped_tabby_cat_with_white_fur_sitting_upright.png"),
)
if OFFICIAL_DEMO_QUICK:
    names = {item.name for item in OFFICIAL_DEMO_RELATIVE_PATHS}
    suffixes = {".png", ".jpg", ".jpeg", ".webp"}
    unexpected = sorted(
        item.name for item in INPUT_DIR.iterdir()
        if item.is_file() and item.suffix.lower() in suffixes
        and item.name not in names
    )
    if unexpected:
        raise RuntimeError(
            "Unexpected images in official-demo input directory: "
            + ", ".join(unexpected)
        )
    for relative in OFFICIAL_DEMO_RELATIVE_PATHS:
        source = COLAB_REPO / relative
        destination = INPUT_DIR / source.name
        source_sha = hashlib.sha256(source.read_bytes()).hexdigest()
        if destination.exists():
            if hashlib.sha256(destination.read_bytes()).hexdigest() != source_sha:
                raise RuntimeError("Drive demo copy has changed: " + str(destination))
            continue
        temporary = destination.with_name(destination.name + ".tmp")
        shutil.copy2(source, temporary)
        os.replace(temporary, destination)
    print("Staged exactly three official demo inputs:")
    for item in sorted(INPUT_DIR.iterdir()):
        if item.is_file() and item.suffix.lower() in suffixes:
            print(" -", item.name)

print("Repository ready:", COLAB_REPO)


In [ ]:
# 4. Install requirements-colab, pytest, and the pinned official MEt3R package.
run_command(
    [sys.executable, "-m", "pip", "install", "-r",
     str(COLAB_REPO / "requirements-colab.txt")]
)
run_command([sys.executable, "-m", "pip", "install", "pytest"])

import importlib.util
MET3R_INSTALL_OK = False
if RUN_MET3R:
    met3r_install = run_command(
        [sys.executable, "-m", "pip", "install",
         "git+https://github.com/mohammadasim98/met3r@" + MET3R_REVISION],
        check=False
    )
    importlib.invalidate_caches()
    MET3R_INSTALL_OK = (
        met3r_install.returncode == 0
        and importlib.util.find_spec("met3r") is not None
    )

required_modules = [
    "torch", "diffusers", "transformers", "huggingface_hub",
    "yaml", "pytest", "jaxtyping"
]
missing_modules = [
    name for name in required_modules if importlib.util.find_spec(name) is None
]
if missing_modules:
    raise RuntimeError("Missing required modules: " + ", ".join(missing_modules))
if RUN_MET3R and not MET3R_INSTALL_OK:
    print("MEt3R is unavailable; strict FULL completion will remain blocked.")
else:
    print("Dependency verification passed; official MEt3R available:",
          MET3R_INSTALL_OK)


In [ ]:
# 5. Record git commit, GPU, CUDA, and disk capacity.
import platform
import torch

git_commit = run_command(
    ["git", "rev-parse", "HEAD"], cwd=COLAB_REPO, capture_output=True
).stdout.strip()
git_status = run_command(
    ["git", "status", "--short"], cwd=COLAB_REPO, capture_output=True
).stdout
run_command(["nvidia-smi"], check=False)

CUDA_READY = bool(torch.cuda.is_available())
gpu_name = torch.cuda.get_device_name(0) if CUDA_READY else None
if gpu_name and "A100" in gpu_name.upper():
    gpu_guidance = "A100 detected: recommended formal 40GB-class runtime."
elif gpu_name and "L4" in gpu_name.upper():
    gpu_guidance = "L4 detected: supported, but slower than A100."
else:
    gpu_guidance = "Recommended formal GPU is A100 40GB; L4 is supported but slower."

def disk_record(path):
    usage = shutil.disk_usage(path)
    return {
        "path": str(path),
        "total_gib": round(usage.total / (1024 ** 3), 2),
        "free_gib": round(usage.free / (1024 ** 3), 2),
    }

environment_record = {
    "git_commit": git_commit,
    "git_status_short": git_status.splitlines(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": CUDA_READY,
    "torch_cuda": torch.version.cuda,
    "gpu": gpu_name,
    "gpu_guidance": gpu_guidance,
    "disk": [disk_record("/content"), disk_record("/content/drive")],
}
parent_environment_dir = OUTPUT_ROOT / "environment"
parent_environment_dir.mkdir(parents=True, exist_ok=True)
PARENT_ENVIRONMENT_FILE = (
    parent_environment_dir / "notebook_environment.json"
)
parent_environment_temporary = PARENT_ENVIRONMENT_FILE.with_name(
    PARENT_ENVIRONMENT_FILE.name + ".tmp"
)
parent_environment_temporary.write_text(
    json.dumps(environment_record, indent=2) + "\n", encoding="utf-8"
)
os.replace(parent_environment_temporary, PARENT_ENVIRONMENT_FILE)
print(json.dumps(environment_record, indent=2))
if not CUDA_READY:
    print("CPU tests/preflight/report can run, but PILOT/FULL are blocked.")


In [ ]:
# 6. Resolve immutable Hugging Face revisions, verify checkpoints/cache, and freeze YAML.
from copy import deepcopy
from huggingface_hub import hf_hub_download, model_info, snapshot_download
from huggingface_hub.utils import HfHubHTTPError
from getpass import getpass
from scripts.run_nile_lowrank_study import experiment_root, load_config, resolve_config
import yaml

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
    except Exception:
        pass
except ImportError:
    pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Using HF_TOKEN from userdata/environment (value not displayed).")
else:
    print("No HF_TOKEN found; using public repositories anonymously.")

def resolve_hf_commit(repo_id, requested_revision):
    global HF_TOKEN

    def fetch_info():
        return model_info(
            repo_id,
            revision=requested_revision or "main",
            token=HF_TOKEN,
        )

    try:
        info = fetch_info()
    except HfHubHTTPError as error:
        status_code = getattr(getattr(error, "response", None), "status_code", None)
        if status_code not in (401, 403):
            raise
        if not HF_TOKEN:
            entered_token = getpass(
                "Hugging Face token required (hidden input): "
            ).strip()
            if entered_token:
                HF_TOKEN = entered_token
                os.environ["HF_TOKEN"] = HF_TOKEN
        if not HF_TOKEN:
            raise RuntimeError(
                "Authentication is required for " + repo_id
                + "; no Hugging Face token was provided."
            ) from error
        try:
            info = fetch_info()
        except HfHubHTTPError as retry_error:
            retry_status = getattr(
                getattr(retry_error, "response", None), "status_code", None
            )
            if retry_status in (401, 403):
                raise RuntimeError(
                    "Hugging Face access to " + repo_id
                    + " was denied. Verify the token and accept any gated "
                    "repository terms in the browser, then rerun this cell."
                ) from retry_error
            raise
    if not info.sha or len(info.sha) != 40:
        raise RuntimeError("Could not resolve immutable revision for " + repo_id)
    return info.sha

def require_frozen_value(actual, expected, label):
    if str(actual) != str(expected):
        raise RuntimeError(
            "Frozen " + label + " differs from this notebook. "
            "Use a new DRIVE_PROJECT_ROOT instead of mutating a resumable run."
        )

template_path = COLAB_REPO / "configs" / "nile_lowrank_full.yaml"
creating_frozen_config = not FROZEN_CONFIG.exists()
if creating_frozen_config:
    requested_config = yaml.safe_load(template_path.read_text(encoding="utf-8"))
    resolved_config = deepcopy(requested_config)
    resolved_config["experiment"]["output_root"] = str(OUTPUT_ROOT)
    resolved_config["experiment"]["resume"] = bool(RESUME)
    resolved_config["experiment"]["run_met3r"] = bool(RUN_MET3R)
    resolved_config["experiment"]["strict_full_requires_met3r"] = bool(FULL_MODE)
    resolved_config["data"]["input_dir"] = str(INPUT_DIR)
    resolved_config["data"]["drive_input_dir"] = str(INPUT_DIR)
    resolved_config["evaluation"]["met3r_revision"] = MET3R_REVISION
    resolved_config["mode"] = {
        "name": RUN_PROFILE,
        "official_demo_only": bool(OFFICIAL_DEMO_QUICK),
        "formal_result": bool(FULL_MODE),
    }
    if OFFICIAL_DEMO_QUICK:
        resolved_config["experiment"]["name"] = "nile_lowrank_official_demo_quick"
        resolved_config["data"].update({
            "pilot_count": 3,
            "full_count": 0,
            "min_distinct_inputs": 3,
        })
        resolved_config["preflight"]["expected_unattainable_count"] = 0
        resolved_config["runtime"]["min_free_disk_gib"] = 10.0
        resolved_config["pilot"].update({
            "seeds": [0],
            "ranks": [8],
            "target_kls": [1.0],
            "rbf_length_scales_deg": [45.0],
            "expected_configs_per_input_seed": 5,
        })
        resolved_config["full"]["seeds"] = [0]

    model_config = resolved_config["model"]
    for revision_field, repo_field in (
        ("base_model_revision", "base_model"),
        ("vae_model_revision", "vae_model"),
        ("adapter_revision", "adapter_path"),
        ("birefnet_revision", "birefnet_model"),
    ):
        model_config[revision_field] = resolve_hf_commit(
            str(model_config[repo_field]),
            model_config.get(revision_field),
        )
    evaluation_config = resolved_config["evaluation"]
    evaluation_config["identity_model_revision"] = resolve_hf_commit(
        str(evaluation_config["identity_model"]),
        evaluation_config.get("identity_model_revision"),
    )
else:
    resolved_config = yaml.safe_load(
        FROZEN_CONFIG.read_text(encoding="utf-8")
    )
    require_frozen_value(
        resolved_config["experiment"]["output_root"],
        OUTPUT_ROOT,
        "experiment.output_root",
    )
    require_frozen_value(
        resolved_config["data"]["input_dir"],
        INPUT_DIR,
        "data.input_dir",
    )
    require_frozen_value(
        resolved_config["data"]["drive_input_dir"],
        INPUT_DIR,
        "data.drive_input_dir",
    )
    require_frozen_value(
        resolved_config["evaluation"].get("met3r_revision"),
        MET3R_REVISION,
        "evaluation.met3r_revision",
    )
    require_frozen_value(
        resolved_config["experiment"].get("run_met3r"),
        RUN_MET3R,
        "experiment.run_met3r",
    )
    require_frozen_value(
        resolved_config["experiment"].get("strict_full_requires_met3r"),
        FULL_MODE,
        "experiment.strict_full_requires_met3r",
    )
    require_frozen_value(
        resolved_config.get("mode", {}).get("name"),
        RUN_PROFILE,
        "mode.name",
    )
    required_frozen_revisions = (
        resolved_config["model"].get("base_model_revision"),
        resolved_config["model"].get("vae_model_revision"),
        resolved_config["model"].get("adapter_revision"),
        resolved_config["model"].get("birefnet_revision"),
        resolved_config["evaluation"].get("identity_model_revision"),
    )
    if any(
        not isinstance(revision, str) or len(revision) != 40
        for revision in required_frozen_revisions
    ):
        raise RuntimeError(
            "Existing frozen config lacks immutable model revisions. "
            "Use a new DRIVE_PROJECT_ROOT; do not re-resolve main in place."
        )
    print("Resume: reusing every previously frozen Hugging Face revision.")

model_config = resolved_config["model"]
evaluation_config = resolved_config["evaluation"]
resolved_revisions = {
    "base_model": {
        "repo_id": str(model_config["base_model"]),
        "revision": str(model_config["base_model_revision"]),
    },
    "vae_model": {
        "repo_id": str(model_config["vae_model"]),
        "revision": str(model_config["vae_model_revision"]),
    },
    "adapter_path": {
        "repo_id": str(model_config["adapter_path"]),
        "revision": str(model_config["adapter_revision"]),
    },
    "birefnet_model": {
        "repo_id": str(model_config["birefnet_model"]),
        "revision": str(model_config["birefnet_revision"]),
    },
    "identity_model": {
        "repo_id": str(evaluation_config["identity_model"]),
        "revision": str(evaluation_config["identity_model_revision"]),
    },
}

adapter_file = hf_hub_download(
    repo_id=str(model_config["adapter_path"]),
    filename=str(model_config["mv_adapter_checkpoint"]),
    revision=str(model_config["adapter_revision"]),
    cache_dir=str(HF_CACHE_DIR / "hub"),
    token=HF_TOKEN,
)
adapter_sha256 = hashlib.sha256(Path(adapter_file).read_bytes()).hexdigest()
frozen_adapter_sha256 = model_config.get("adapter_sha256")
if frozen_adapter_sha256 and frozen_adapter_sha256 != adapter_sha256:
    raise RuntimeError("Frozen adapter SHA-256 does not match the cached file.")
model_config["adapter_sha256"] = adapter_sha256

snapshot_paths = {}
if PREFETCH_MODEL_SNAPSHOTS:
    ignore_patterns = [
        "*.msgpack", "*.h5", "*.onnx", "*.tflite", "*.ckpt",
        "sd_xl_base_1.0*.safetensors",
    ]
    for label, entry in resolved_revisions.items():
        if label == "adapter_path":
            continue
        if OFFICIAL_DEMO_QUICK and label == "identity_model":
            continue
        snapshot_paths[label] = snapshot_download(
            repo_id=entry["repo_id"],
            revision=entry["revision"],
            cache_dir=str(HF_CACHE_DIR / "hub"),
            token=HF_TOKEN,
            ignore_patterns=ignore_patterns,
        )

if creating_frozen_config:
    frozen_temporary = FROZEN_CONFIG.with_name(
        FROZEN_CONFIG.name + ".tmp"
    )
    frozen_temporary.write_text(
        yaml.safe_dump(
            resolved_config, sort_keys=False, allow_unicode=True
        ),
        encoding="utf-8",
    )
    os.replace(frozen_temporary, FROZEN_CONFIG)

resolved_for_runner = resolve_config(
    load_config(FROZEN_CONFIG),
    input_dir=INPUT_DIR,
    output_root=OUTPUT_ROOT,
)
RUNNER_ARTIFACT_ROOT = experiment_root(resolved_for_runner)
RUNNER_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def atomic_copy_file(source, destination):
    source = Path(source)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    temporary.write_bytes(source.read_bytes())
    os.replace(temporary, destination)

checkpoint_payload = {
    "schema_version": 1,
    "git_commit": git_commit,
    "hf_cache_dir": str(HF_CACHE_DIR),
    "resolved_revisions": resolved_revisions,
    "adapter_checkpoint": {
        "path": str(adapter_file),
        "sha256": adapter_sha256,
    },
    "snapshot_paths": snapshot_paths,
    "met3r_revision": MET3R_REVISION,
    "token_source": "userdata_or_environment" if HF_TOKEN else "anonymous",
}
checkpoint_temporary = CHECKPOINT_MANIFEST.with_name(
    CHECKPOINT_MANIFEST.name + ".tmp"
)
checkpoint_temporary.write_text(
    json.dumps(checkpoint_payload, indent=2) + "\n", encoding="utf-8"
)
os.replace(checkpoint_temporary, CHECKPOINT_MANIFEST)
atomic_copy_file(
    CHECKPOINT_MANIFEST,
    RUNNER_ARTIFACT_ROOT / "configs" / "checkpoint_manifest.json",
)
atomic_copy_file(
    PARENT_ENVIRONMENT_FILE,
    RUNNER_ARTIFACT_ROOT / "environment" / "notebook_environment.json",
)

print("Run profile:", RUN_PROFILE)
print("Frozen config:", FROZEN_CONFIG)
print("Runner artifact root:", RUNNER_ARTIFACT_ROOT)
print("Adapter SHA-256:", adapter_sha256)
print("MEt3R revision:", MET3R_REVISION)
print(json.dumps(resolved_revisions, indent=2))


In [ ]:
# 7. Validate distinct inputs and display the deterministic contact sheet.
input_validation_dir = RUNNER_ARTIFACT_ROOT / "inputs"
input_validation_dir.mkdir(parents=True, exist_ok=True)
validation_command = [
    sys.executable, "-m", "scripts.validate_nile_inputs",
    "--input-dir", str(INPUT_DIR),
    "--drive-input-dir", str(INPUT_DIR),
    "--output-dir", str(input_validation_dir),
    "--pilot-count", str(resolved_config["data"]["pilot_count"]),
    "--full-count", str(resolved_config["data"]["full_count"]),
    "--min-distinct-inputs",
    str(resolved_config["data"]["min_distinct_inputs"]),
]
run_command(validation_command, cwd=COLAB_REPO, check=False)
input_validation = json.loads(
    (input_validation_dir / "input_validation.json").read_text(encoding="utf-8")
)
INPUTS_READY = bool(input_validation.get("formal_ready", False))
print(json.dumps(
    {k: v for k, v in input_validation.items() if k != "records"}, indent=2
))

from IPython.display import Image as IPyImage, display
contact_sheet = input_validation_dir / "input_contact_sheet.jpg"
if contact_sheet.exists():
    display(IPyImage(filename=str(contact_sheet)))
if not INPUTS_READY:
    print("Generation is blocked; tests, preflight, and report will continue.")
elif OFFICIAL_DEMO_QUICK:
    print("Official demo inputs are ready for the 15-run quick matrix.")
    print("These curated images do not constitute a formal FULL dataset.")


In [ ]:
# 8. Run compileall and pytest; atomically persist the exact test status.
from datetime import datetime, timezone

compile_command = [
    sys.executable, "-m", "compileall", "mvadapter", "scripts"
]
known_preexisting_nodeid = (
    "tests/test_nile_workflows.py::"
    "TestDistributionPreservingNotebookWorkflow::"
    "test_notebook_is_clean_and_unexecuted"
)
pytest_command = [
    sys.executable, "-m", "pytest", "-q",
    "--deselect", known_preexisting_nodeid,
]
full_pytest_command = [sys.executable, "-m", "pytest", "-q"]
compile_result = run_command(compile_command, cwd=COLAB_REPO, check=False)
pytest_result = run_command(pytest_command, cwd=COLAB_REPO, check=False)
full_pytest_result = run_command(
    full_pytest_command, cwd=COLAB_REPO, check=False
)
test_status = {
    "schema_version": 1,
    "passed": (
        compile_result.returncode == 0 and pytest_result.returncode == 0
    ),
    "tests_complete": (
        compile_result.returncode == 0 and pytest_result.returncode == 0
    ),
    "compileall_returncode": compile_result.returncode,
    "pytest_returncode": pytest_result.returncode,
    "finished_at": datetime.now(timezone.utc).isoformat(),
    "command": [compile_command, pytest_command],
    "full_repository_pytest_returncode": full_pytest_result.returncode,
    "known_preexisting_failures": (
        [{
            "nodeid": known_preexisting_nodeid,
            "reason": (
                "The retained distribution-preserving notebook contains "
                "intentional execution evidence and is not cleaned here."
            ),
        }]
        if full_pytest_result.returncode != 0 else []
    ),
}
environment_dir = RUNNER_ARTIFACT_ROOT / "environment"
environment_dir.mkdir(parents=True, exist_ok=True)
atomic_copy_file(
    PARENT_ENVIRONMENT_FILE,
    environment_dir / "notebook_environment.json",
)
for filename in ("test_results.json", "test_status.json"):
    destination = environment_dir / filename
    temporary = destination.with_name(destination.name + ".tmp")
    temporary.write_text(
        json.dumps(test_status, indent=2) + "\n", encoding="utf-8"
    )
    os.replace(temporary, destination)
print(json.dumps(test_status, indent=2))
if not test_status["passed"]:
    print(
        "Compile or test gate failed. The receipt remains false; "
        "PILOT/FULL will be blocked, while preflight/report continue."
    )


In [ ]:
# 9. Run the CPU preflight and all distribution gates before GPU generation.
run_stage("preflight")
gate_path = (
    RUNNER_ARTIFACT_ROOT
    / "distribution_gates"
    / "configuration_gates.json"
)
if gate_path.exists():
    gate_payload = json.loads(gate_path.read_text(encoding="utf-8"))
    print("Global distribution gate:", gate_payload.get("passed"))
    gate_rows = gate_payload.get("configurations", [])
    if gate_rows:
        import pandas as pd
        gate_columns = [
            key for key in (
                "config_id", "method", "rank", "target_kl", "alpha",
                "achieved_kl", "passed", "exclusion_reason"
            ) if any(key in row for row in gate_rows)
        ]
        display(pd.DataFrame(gate_rows)[gate_columns])
else:
    print("Preflight artifact missing; generation remains blocked.")


In [ ]:
# 10. PILOT generation: dry-run the plan without touching its manifest, display the estimate, then run or resume.
pilot_manifest_path = RUNNER_ARTIFACT_ROOT / "pilot" / "manifest.json"
pilot_manifest_before = (
    pilot_manifest_path.read_bytes() if pilot_manifest_path.exists() else None
)
pilot_preview_result = run_stage(
    "pilot", dry_run=True, capture_output=True
)
pilot_preview_payload = json.loads(pilot_preview_result.stdout)
pilot_preview = pilot_preview_payload["status"]["stages"]["pilot"]
print(json.dumps({
    "planned": pilot_preview.get("planned"),
    "would_execute": pilot_preview.get("would_execute"),
    "estimated_output_bytes": pilot_preview.get("estimated_output_bytes"),
}, indent=2))
pilot_manifest_after = (
    pilot_manifest_path.read_bytes() if pilot_manifest_path.exists() else None
)
if pilot_manifest_after != pilot_manifest_before:
    raise RuntimeError("PILOT dry-run changed the formal manifest.")
pilot_run_result = run_stage("pilot", capture_output=True)
if pilot_manifest_path.exists():
    pilot_manifest = json.loads(
        pilot_manifest_path.read_text(encoding="utf-8")
    )
    pilot_records = (
        pilot_manifest.get("runs", [])
        if isinstance(pilot_manifest, dict) else pilot_manifest
    )
    import pandas as pd
    pilot_status = pd.DataFrame(pilot_records)
    if not pilot_status.empty and "status" in pilot_status:
        display(
            pilot_status.groupby(["method", "status"]).size()
            .rename("runs").reset_index()
        )
        failed_rows = pilot_status[pilot_status["status"] == "failed"]
        if not failed_rows.empty:
            diagnostic_columns = [
                key for key in (
                    "run_id", "method", "error", "traceback",
                    "oom", "worker_returncode", "log_path"
                ) if key in failed_rows.columns
            ]
            print("PILOT failure diagnostics:")
            display(failed_rows[diagnostic_columns].drop_duplicates())
            runtime_path = RUNNER_ARTIFACT_ROOT / "runtime_status.json"
            if runtime_path.exists():
                runtime_payload = json.loads(
                    runtime_path.read_text(encoding="utf-8")
                )
                pilot_stage = (
                    runtime_payload.get("stages", {}).get("pilot", {})
                )
                fatal_event = pilot_stage.get("worker_fatal_event")
                if fatal_event:
                    print("Worker fatal event:")
                    print(json.dumps(fatal_event, indent=2, ensure_ascii=False))
                worker_log_path = pilot_stage.get("worker_log_path")
                if worker_log_path and Path(worker_log_path).exists():
                    print("Worker log tail:")
                    print(Path(worker_log_path).read_text(
                        encoding="utf-8", errors="replace"
                    )[-12000:])
else:
    print("PILOT manifest absent; inspect runtime_status.json.")


In [ ]:
# 11. Run official MEt3R and guardrail evaluation for PILOT.
# QUICK_DEMO skips this expensive evaluator; FORMAL_FULL requires it.
run_stage("evaluate", enabled=RUN_MET3R)
pilot_metrics_path = (
    RUNNER_ARTIFACT_ROOT / "metrics" / "pilot" / "lowrank_metrics.json"
)
if pilot_metrics_path.exists():
    pilot_metrics = json.loads(
        pilot_metrics_path.read_text(encoding="utf-8")
    )
    summaries = pilot_metrics.get("configuration_summaries", [])
    if summaries:
        import pandas as pd
        display(pd.DataFrame(summaries))
elif OFFICIAL_DEMO_QUICK:
    print("MEt3R intentionally skipped in OFFICIAL_DEMO_QUICK.")
else:
    print("PILOT MEt3R unavailable; candidate selection stays blocked.")


In [ ]:
# 12. Select one candidate per topology and freeze the immutable selection.
run_stage("select", enabled=FULL_MODE and RUN_MET3R)
selection_json = (
    RUNNER_ARTIFACT_ROOT
    / "selected_candidates"
    / "selected_candidates.json"
)
selection_yaml = (
    RUNNER_ARTIFACT_ROOT
    / "selected_candidates"
    / "selected_candidates.yaml"
)
if selection_json.exists():
    selected_candidates = json.loads(
        selection_json.read_text(encoding="utf-8")
    )
    print("Selection hash:",
          selected_candidates.get("configuration_hash"))
else:
    selected_candidates = None
    if OFFICIAL_DEMO_QUICK:
        print("Candidate freezing intentionally skipped in OFFICIAL_DEMO_QUICK.")
    else:
        print("No eligible frozen selection; FULL cannot start.")


In [ ]:
# 13. Display the frozen candidate configuration (no manual run id).
from IPython.display import Code
if selection_yaml.exists():
    display(Code(selection_yaml.read_text(encoding="utf-8"), language="yaml"))
elif selection_json.exists():
    display(Code(selection_json.read_text(encoding="utf-8"), language="json"))
elif OFFICIAL_DEMO_QUICK:
    print("No frozen candidate is expected for the visual quick profile.")
else:
    print("Frozen configuration unavailable; inspect PILOT blockers.")


In [ ]:
# 14. Run or resume the independent FULL matrix on the disjoint FULL split.
run_stage("full", enabled=FULL_MODE)
full_manifest_path = RUNNER_ARTIFACT_ROOT / "full" / "manifest.json"
if full_manifest_path.exists():
    full_manifest = json.loads(
        full_manifest_path.read_text(encoding="utf-8")
    )
    full_records = (
        full_manifest.get("runs", [])
        if isinstance(full_manifest, dict) else full_manifest
    )
    import pandas as pd
    full_status = pd.DataFrame(full_records)
    if not full_status.empty and "status" in full_status:
        display(
            full_status.groupby(["method", "status"]).size()
            .rename("runs").reset_index()
        )
else:
    print("FULL manifest absent; inspect runtime_status.json.")


In [ ]:
# 15. Run read-only paired trajectory observers and summarize survival.
run_stage("trajectory", enabled=FULL_MODE)
trajectory_summary_path = (
    RUNNER_ARTIFACT_ROOT / "trajectory" / "trajectory_summary.json"
)
if trajectory_summary_path.exists():
    trajectory_summary = json.loads(
        trajectory_summary_path.read_text(encoding="utf-8")
    )
    print(json.dumps({
        "complete": trajectory_summary.get("complete"),
        "correlation_state": trajectory_summary.get("correlation_state"),
        "pair_count": trajectory_summary.get("pair_count"),
        "failure_count": trajectory_summary.get("failure_count"),
        "aggregate_final_g_t": trajectory_summary.get("aggregate_final_g_t"),
    }, indent=2))
else:
    print("Trajectory summary unavailable.")


In [ ]:
# 16. Evaluate PILOT/FULL, compute paired statistics, CIs, and Holm tests.
run_stage("evaluate", enabled=RUN_MET3R)
full_metrics_path = (
    RUNNER_ARTIFACT_ROOT / "metrics" / "full" / "lowrank_metrics.json"
)
for label, metrics_path in (
    ("PILOT", pilot_metrics_path), ("FULL", full_metrics_path)
):
    if not metrics_path.exists():
        print(label, "metrics unavailable")
        continue
    metrics_payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    statistics_rows = metrics_payload.get("paired_statistics", [])
    angle_rows = metrics_payload.get("angle_bin_summaries", [])
    print(label, "paired statistics")
    if statistics_rows:
        import pandas as pd
        display(pd.DataFrame(statistics_rows))
    if angle_rows:
        display(pd.DataFrame(angle_rows))


In [ ]:
# 17. Generate the evidence-bounded report and FINAL_STATUS.json.
run_stage("report")
report_path = RUNNER_ARTIFACT_ROOT / "FULL_EXPERIMENT_REPORT.md"
report_json_path = RUNNER_ARTIFACT_ROOT / "FULL_EXPERIMENT_REPORT.json"
final_status_path = RUNNER_ARTIFACT_ROOT / "FINAL_STATUS.json"
reproduce_path = RUNNER_ARTIFACT_ROOT / "REPRODUCE.md"
print("Report:", report_path)
print("Final status:", final_status_path)
print("Reproduction guide:", reproduce_path)


In [ ]:
# 18. Display core tables, diagnostic figures, report, and FINAL_STATUS.
from IPython.display import Markdown

for label, metrics_path in (
    ("PILOT", pilot_metrics_path), ("FULL", full_metrics_path)
):
    if not metrics_path.exists():
        continue
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    rows = payload.get("configuration_summaries", [])
    if rows:
        import pandas as pd
        frame = pd.DataFrame(rows)
        preferred = [
            column for column in (
                "method", "rank", "target_kl", "alpha",
                "met3r_all_pair_mean", "met3r_standard_error",
                "dino_identity_mean_delta", "small_component_ratio_mean",
                "foreground_area_cv_mean", "r_hf"
            ) if column in frame.columns
        ]
        print(label, "core configuration table")
        display(frame[preferred] if preferred else frame)

if OFFICIAL_DEMO_QUICK and pilot_manifest_path.exists():
    quick_payload = json.loads(
        pilot_manifest_path.read_text(encoding="utf-8")
    )
    quick_records = [
        row for row in quick_payload.get("runs", [])
        if row.get("status") == "succeeded" and Path(row["output"]).exists()
    ]
    print("Official demo quick results:", len(quick_records), "/ 15")
    if quick_records:
        import pandas as pd
        quick_frame = pd.DataFrame(quick_records)
        columns = [
            key for key in (
                "input_path", "method", "rank", "target_kl",
                "rbf_length_scale_deg", "alpha", "status", "output"
            ) if key in quick_frame.columns
        ]
        display(quick_frame[columns])
        method_order = {
            "iid_external": 0,
            "shared_full": 1,
            "lowrank_camera_rbf": 2,
            "lowrank_nested_tree_a": 3,
            "lowrank_nested_tree_ab": 4,
        }
        for row in sorted(
            quick_records,
            key=lambda item: (
                Path(item["input_path"]).name,
                method_order.get(item["method"], 99),
            ),
        ):
            print(
                Path(row["input_path"]).name,
                "|", row["method"],
                "| rank", row.get("rank"),
                "| KL", row.get("target_kl"),
+                "| alpha", row.get("alpha"),
            )
            display(IPyImage(filename=row["output"]))

diagnostics_dir = (
    RUNNER_ARTIFACT_ROOT / "distribution_gates" / "diagnostics"
)
figure_candidates = [
    diagnostics_dir / "configuration_gate_matrix.png",
    diagnostics_dir / "alpha_vs_achieved_kl.png",
]
figure_candidates.extend(sorted(diagnostics_dir.glob("*.png"))[:8])
figure_candidates.extend(
    sorted((RUNNER_ARTIFACT_ROOT / "trajectory").glob("**/*.png"))[:6]
)
for figure_path in figure_candidates:
    if figure_path.exists():
        print(figure_path.relative_to(RUNNER_ARTIFACT_ROOT))
        display(IPyImage(filename=str(figure_path)))

if report_path.exists():
    display(Markdown(report_path.read_text(encoding="utf-8")))
if final_status_path.exists():
    FINAL_STATUS = json.loads(
        final_status_path.read_text(encoding="utf-8")
    )
    if OFFICIAL_DEMO_QUICK:
        print(
            "Expected: FULL remains incomplete in this visual-only quick profile."
        )
    print(json.dumps(FINAL_STATUS, indent=2, ensure_ascii=False))
else:
    raise RuntimeError("FINAL_STATUS.json missing; inspect report output.")
